In [5]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# SETTINGS
# =========================================================
BASE_PRED_DIR = r"C:\IDEAL_Programming\Cases\UI_Results"
REAL_PATH = r"C:\IDEAL_Programming\Cases\tight_case_season_real_profiles.csv"

CASE_FOLDER_MAP = {
    "Case_1": "case_1_flat_2_medium",
    "Case_2": "case_2_flat_1_small",
    "Case_3": "case_3_flat_3_medium",
    "Case_4": "case_4_house_4_large",
    "Case_5": "case_5_house_2_medium",
    "Case_6": "case_6_house_3_large",
}

MODEL_COLUMNS = {
    "rf": "rf_Wh",
    "xgb": "xgb_Wh",
    "lgbm": "lgbm_Wh",
    "ensemble": "ensemble_Wh"
}

SEASON_DATE_MAP = {
    "01-28": "winter",
    "04-09": "spring",
    "07-28": "summer",
    "10-30": "autumn"
}

SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

# =========================================================
# HELPERS
# =========================================================
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    mask = np.abs(y_true) > 1e-6
    if mask.sum() == 0:
        mape = np.nan
    else:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    return mae, rmse, mape


def detect_season_from_filename(filename):
    for md, season in SEASON_DATE_MAP.items():
        if md in filename:
            return season, md
    return None, None


def load_real_profile(real_df, case_name, season):
    temp = real_df[
        (real_df["case"] == case_name) &
        (real_df["season"] == season)
    ].sort_values("hour")

    if temp.empty:
        return None

    return temp["real_consumption_Wh"].to_numpy(dtype=float)


def save_plot(hours, real, pred_dict, season, out_path, case_name):
    plt.figure(figsize=(10, 5))
    plt.plot(hours, real, label="Real", linewidth=2)

    for model_name, pred in pred_dict.items():
        plt.plot(hours, pred, label=model_name)

    plt.title(f"{case_name} - {season}")
    plt.xlabel("Hour")
    plt.ylabel("Consumption (Wh)")
    plt.xticks(hours)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def evaluate_single_case(pred_dir, case_name, real_df):
    pred_files = sorted(glob.glob(os.path.join(pred_dir, "*.csv")))

    if not pred_files:
        print(f"No CSV files found in: {pred_dir}")
        return None, None, None

    # Φιλτράρουμε μόνο τα prediction files
    pred_files = [f for f in pred_files if os.path.basename(f).startswith("ui_pred_")]

    if not pred_files:
        print(f"No prediction files found in: {pred_dir}")
        return None, None, None

    all_metric_rows = []
    season_plot_data = {}

    for pred_file in pred_files:
        fname = os.path.basename(pred_file)
        season, month_day = detect_season_from_filename(fname)

        if season is None:
            print(f"Skipping file (season not recognized): {fname}")
            continue

        pred_df = pd.read_csv(pred_file)

        if "timestamp" in pred_df.columns:
            pred_df["timestamp"] = pd.to_datetime(pred_df["timestamp"], errors="coerce")
            pred_df = pred_df.sort_values("timestamp").reset_index(drop=True)

        real_profile = load_real_profile(real_df, case_name, season)
        if real_profile is None:
            print(f"No real profile found for {case_name} - {season}")
            continue

        if len(real_profile) != 24:
            print(f"Real profile for {case_name} - {season} does not have 24 rows. Found {len(real_profile)}")
            continue

        pred_dict_for_plot = {}

        for model_name, col in MODEL_COLUMNS.items():
            if col not in pred_df.columns:
                print(f"Column {col} not found in {fname}, skipping model {model_name}")
                continue

            pred_values = pred_df[col].to_numpy(dtype=float)

            if len(pred_values) != 24:
                print(f"{fname} - {col} does not have 24 rows. Found {len(pred_values)}")
                continue

            mae, rmse, mape = compute_metrics(real_profile, pred_values)

            all_metric_rows.append({
                "case": case_name,
                "season": season,
                "month_day": month_day,
                "source_file": fname,
                "model": model_name,
                "MAE": mae,
                "RMSE": rmse,
                "MAPE": mape,
                "pred_total_kWh": pred_values.sum() / 1000.0,
                "real_total_kWh": real_profile.sum() / 1000.0
            })

            pred_dict_for_plot[model_name] = pred_values

        if pred_dict_for_plot:
            hours = list(range(24))
            plot_path = os.path.join(pred_dir, f"{case_name}_{season}_comparison.png")
            save_plot(hours, real_profile, pred_dict_for_plot, season, plot_path, case_name)

            season_plot_data[season] = {
                "hours": hours,
                "real": real_profile,
                "preds": pred_dict_for_plot
            }

    if not all_metric_rows:
        return None, None, None

    metrics_df = pd.DataFrame(all_metric_rows)
    metrics_df["season"] = pd.Categorical(metrics_df["season"], categories=SEASON_ORDER, ordered=True)
    metrics_df = metrics_df.sort_values(["season", "model"]).reset_index(drop=True)

    summary_model_df = (
        metrics_df.groupby("model", as_index=False)
        .agg(
            mean_MAE=("MAE", "mean"),
            mean_RMSE=("RMSE", "mean"),
            mean_MAPE=("MAPE", "mean"),
            mean_pred_total_kWh=("pred_total_kWh", "mean"),
            mean_real_total_kWh=("real_total_kWh", "mean")
        )
        .sort_values("mean_RMSE")
        .reset_index(drop=True)
    )

    summary_season_model_df = metrics_df.pivot_table(
        index="season",
        columns="model",
        values=["MAE", "RMSE", "MAPE"],
        aggfunc="mean"
    )

    # Save metrics for this case
    metrics_path = os.path.join(pred_dir, f"{case_name}_model_metrics_detailed.csv")
    summary_model_path = os.path.join(pred_dir, f"{case_name}_model_metrics_summary.csv")
    summary_season_model_path = os.path.join(pred_dir, f"{case_name}_metrics_by_season_and_model.csv")

    metrics_df.to_csv(metrics_path, index=False)
    summary_model_df.to_csv(summary_model_path, index=False)
    summary_season_model_df.to_csv(summary_season_model_path)

    # Combined plot per model across seasons
    for model_name in MODEL_COLUMNS.keys():
        available_seasons = [
            s for s in SEASON_ORDER
            if s in season_plot_data and model_name in season_plot_data[s]["preds"]
        ]
        if not available_seasons:
            continue

        plt.figure(figsize=(12, 8))

        for i, season in enumerate(available_seasons, start=1):
            plt.subplot(2, 2, i)
            hours = season_plot_data[season]["hours"]
            real_vals = season_plot_data[season]["real"]
            pred_vals = season_plot_data[season]["preds"][model_name]

            plt.plot(hours, real_vals, label="Real", linewidth=2)
            plt.plot(hours, pred_vals, label=model_name)
            plt.title(f"{season}")
            plt.xlabel("Hour")
            plt.ylabel("Wh")
            plt.xticks(hours)
            plt.grid(True, alpha=0.3)
            plt.legend()

        plt.suptitle(f"{case_name} - {model_name} vs Real Across Seasons")
        plt.tight_layout(rect=[0, 0, 1, 0.96])

        combined_plot_path = os.path.join(pred_dir, f"{case_name}_{model_name}_all_seasons.png")
        plt.savefig(combined_plot_path, dpi=200)
        plt.close()

    print("=" * 90)
    print(f"CASE: {case_name}")
    print("=" * 90)
    print(summary_model_df)
    print("\nSaved files:")
    print(metrics_path)
    print(summary_model_path)
    print(summary_season_model_path)
    print(f"Plots saved in: {pred_dir}")
    print()

    return metrics_df, summary_model_df, summary_season_model_df


# =========================================================
# MAIN
# =========================================================
real_df = pd.read_csv(REAL_PATH)

all_cases_metrics = []
all_cases_summary = []

for folder_name, case_name in CASE_FOLDER_MAP.items():
    pred_dir = os.path.join(BASE_PRED_DIR, folder_name)

    if not os.path.isdir(pred_dir):
        print(f"Folder not found, skipping: {pred_dir}")
        continue

    metrics_df, summary_model_df, _ = evaluate_single_case(pred_dir, case_name, real_df)

    if metrics_df is not None:
        all_cases_metrics.append(metrics_df)

    if summary_model_df is not None:
        tmp = summary_model_df.copy()
        tmp.insert(0, "case", case_name)
        all_cases_summary.append(tmp)

# =========================================================
# SAVE GLOBAL SUMMARIES
# =========================================================
if all_cases_metrics:
    all_metrics_df = pd.concat(all_cases_metrics, ignore_index=True)
    all_metrics_path = os.path.join(BASE_PRED_DIR, "all_cases_model_metrics_detailed.csv")
    all_metrics_df.to_csv(all_metrics_path, index=False)
else:
    all_metrics_df = None
    all_metrics_path = None

if all_cases_summary:
    all_summary_df = pd.concat(all_cases_summary, ignore_index=True)
    all_summary_path = os.path.join(BASE_PRED_DIR, "all_cases_model_metrics_summary.csv")
    all_summary_df.to_csv(all_summary_path, index=False)
else:
    all_summary_df = None
    all_summary_path = None

# =========================================================
# PRINT GLOBAL RESULTS
# =========================================================
print("=" * 90)
print("GLOBAL RESULTS")
print("=" * 90)

if all_metrics_path:
    print(f"Detailed metrics for all cases saved to: {all_metrics_path}")
else:
    print("No detailed metrics were produced.")

if all_summary_path:
    print(f"Summary metrics for all cases saved to: {all_summary_path}")
    print("\nALL CASES SUMMARY:")
    print(all_summary_df)
else:
    print("No summary metrics were produced.")

C:\Users\z0050azt\AppData\Local\Temp\ipykernel_24656\335182365.py:195: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  summary_season_model_df = metrics_df.pivot_table(


CASE: case_1_flat_2_medium
      model   mean_MAE   mean_RMSE  mean_MAPE  mean_pred_total_kWh  \
0        rf  67.707487   99.821617  30.345529             4.110115   
1  ensemble  64.630604  100.720824  26.792832             3.932039   
2      lgbm  64.127555  102.853137  25.981813             3.927387   
3       xgb  65.993601  107.089891  26.218951             3.699257   

   mean_real_total_kWh  
0             4.834571  
1             4.834571  
2             4.834571  
3             4.834571  

Saved files:
C:\IDEAL_Programming\Cases\UI_Results\Case_1\case_1_flat_2_medium_model_metrics_detailed.csv
C:\IDEAL_Programming\Cases\UI_Results\Case_1\case_1_flat_2_medium_model_metrics_summary.csv
C:\IDEAL_Programming\Cases\UI_Results\Case_1\case_1_flat_2_medium_metrics_by_season_and_model.csv
Plots saved in: C:\IDEAL_Programming\Cases\UI_Results\Case_1



C:\Users\z0050azt\AppData\Local\Temp\ipykernel_24656\335182365.py:195: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  summary_season_model_df = metrics_df.pivot_table(


CASE: case_2_flat_1_small
      model   mean_MAE  mean_RMSE  mean_MAPE  mean_pred_total_kWh  \
0      lgbm  57.552088  94.624968  26.172102             3.601743   
1       xgb  56.724245  95.779472  24.525976             3.506824   
2  ensemble  57.081058  96.265156  25.159818             3.546178   
3        rf  58.644702  99.541935  25.679991             3.534020   

   mean_real_total_kWh  
0             4.456042  
1             4.456042  
2             4.456042  
3             4.456042  

Saved files:
C:\IDEAL_Programming\Cases\UI_Results\Case_2\case_2_flat_1_small_model_metrics_detailed.csv
C:\IDEAL_Programming\Cases\UI_Results\Case_2\case_2_flat_1_small_model_metrics_summary.csv
C:\IDEAL_Programming\Cases\UI_Results\Case_2\case_2_flat_1_small_metrics_by_season_and_model.csv
Plots saved in: C:\IDEAL_Programming\Cases\UI_Results\Case_2



C:\Users\z0050azt\AppData\Local\Temp\ipykernel_24656\335182365.py:195: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  summary_season_model_df = metrics_df.pivot_table(


...

CASE: case_4_house_4_large
      model    mean_MAE   mean_RMSE  mean_MAPE  mean_pred_total_kWh  \
0       xgb  163.219445  199.623278  64.867527            11.574198   
1      lgbm  178.433905  219.862305  63.225174            10.449752   
2  ensemble  192.924232  236.897014  57.521714             9.322144   
3        rf  244.926779  303.252169  52.470888             6.787397   

   mean_real_total_kWh  
0            11.319745  
1            11.319745  
2            11.319745  
3            11.319745  

Saved files:
C:\IDEAL_Programming\Cases\UI_Results\Case_4\case_4_house_4_large_model_metrics_detailed.csv
C:\IDEAL_Programming\Cases\UI_Results\Case_4\case_4_house_4_large_model_metrics_summary.csv
C:\IDEAL_Programming\Cases\UI_Results\Case_4\case_4_house_4_large_metrics_by_season_and_model.csv
Plots saved in: C:\IDEAL_Programming\Cases\UI_Results\Case_4



C:\Users\z0050azt\AppData\Local\Temp\ipykernel_24656\335182365.py:195: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  summary_season_model_df = metrics_df.pivot_table(


CASE: case_5_house_2_medium
      model    mean_MAE   mean_RMSE  mean_MAPE  mean_pred_total_kWh  \
0        rf  131.150280  172.881671  26.100556             7.058940   
1  ensemble  156.735622  194.262934  32.576235             6.330644   
2      lgbm  173.682586  208.362274  37.102112             5.917385   
3       xgb  179.056050  213.602161  38.366894             5.772844   

   mean_real_total_kWh  
0            10.070189  
1            10.070189  
2            10.070189  
3            10.070189  

Saved files:
C:\IDEAL_Programming\Cases\UI_Results\Case_5\case_5_house_2_medium_model_metrics_detailed.csv
C:\IDEAL_Programming\Cases\UI_Results\Case_5\case_5_house_2_medium_model_metrics_summary.csv
C:\IDEAL_Programming\Cases\UI_Results\Case_5\case_5_house_2_medium_metrics_by_season_and_model.csv
Plots saved in: C:\IDEAL_Programming\Cases\UI_Results\Case_5



C:\Users\z0050azt\AppData\Local\Temp\ipykernel_24656\335182365.py:195: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  summary_season_model_df = metrics_df.pivot_table(


CASE: case_6_house_3_large
      model    mean_MAE   mean_RMSE  mean_MAPE  mean_pred_total_kWh  \
0       xgb  133.490396  173.808252  26.938112            10.263622   
1      lgbm  148.554883  200.054376  26.881828             8.820641   
2  ensemble  152.953653  203.066771  27.830733             8.783966   
3        rf  181.180001  238.787904  31.704991             7.646717   

   mean_real_total_kWh  
0            11.559125  
1            11.559125  
2            11.559125  
3            11.559125  

Saved files:
C:\IDEAL_Programming\Cases\UI_Results\Case_6\case_6_house_3_large_model_metrics_detailed.csv
C:\IDEAL_Programming\Cases\UI_Results\Case_6\case_6_house_3_large_model_metrics_summary.csv
C:\IDEAL_Programming\Cases\UI_Results\Case_6\case_6_house_3_large_metrics_by_season_and_model.csv
Plots saved in: C:\IDEAL_Programming\Cases\UI_Results\Case_6

GLOBAL RESULTS
Detailed metrics for all cases saved to: C:\IDEAL_Programming\Cases\UI_Results\all_cases_model_metrics_detailed.csv
Su